In [1]:
import pandas as pd
import re
from pathlib import Path

# Source folder containing the original Wastes Removed files
source_folder = Path("/Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Wastes Removed")

# Output folder for the merged file
output_folder = Path("/Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Data_clean")
output_folder.mkdir(parents=True, exist_ok=True)

# Find matching Wastes Removed .xlsb files
excel_files = list(source_folder.glob("*Waste Data Interrogator - Wastes Removed*Version *.xlsb"))

print("Found Wastes Removed files:")
for file in excel_files:
    print("-", file.name)

if not excel_files:
    raise FileNotFoundError("No matching Wastes Removed .xlsb files were found in the source folder.")

dataframes = []

for file_path in excel_files:
    file_name = file_path.name

    # Extract year from file name
    year_match = re.search(r"\b(20\d{2})\b", file_name)

    if not year_match:
        print(f"Skipped file because no year was found in the file name: {file_name}")
        continue

    year = int(year_match.group(1))

    # Expected sheet name based on year
    expected_sheet_name = f"{year} Waste Removed"

    # Open .xlsb file
    excel_file = pd.ExcelFile(file_path, engine="pyxlsb")

    print(f"\nProcessing file: {file_name}")
    print("Available sheets:", excel_file.sheet_names)

    # Select the correct sheet
    if expected_sheet_name in excel_file.sheet_names:
        sheet_name = expected_sheet_name
    else:
        possible_sheets = [
            sheet for sheet in excel_file.sheet_names
            if "waste removed" in sheet.lower()
        ]

        if not possible_sheets:
            raise ValueError(f"No 'Waste Removed' sheet was found in file: {file_name}")

        sheet_name = possible_sheets[0]

    print(f"Reading sheet: {sheet_name}")

    # Read data from the selected sheet
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine="pyxlsb"
    )

    # Add year column
    df["year"] = year

    dataframes.append(df)

if not dataframes:
    raise ValueError("No data was available to merge.")

# Merge all dataframes
merged_df = pd.concat(dataframes, ignore_index=True)

# Export merged file to the output folder
output_file = output_folder / "Merged_Waste_Data_Removed_2023_2024.xlsx"

merged_df.to_excel(
    output_file,
    index=False,
    engine="openpyxl"
)

print("\nMerge completed successfully.")
print(f"Output file: {output_file}")
print(f"Total rows: {len(merged_df)}")
print(f"Total columns: {len(merged_df.columns)}")

Found Wastes Removed files:
- 2023 Waste Data Interrogator - Wastes Removed (Excel) - Version 1.xlsb
- 2024 Waste Data Interrogator - Wastes Removed (Excel) - Version 2.xlsb

Processing file: 2023 Waste Data Interrogator - Wastes Removed (Excel) - Version 1.xlsb
Available sheets: ['Information', 'Glossary', '2023 Waste Removed', 'Interrogator - Waste Removed', 'Active Sites', 'Configurable Reports', 'One Click Summaries', 'Waste Data Links', 'European Waste Catalogue']
Reading sheet: 2023 Waste Removed

Processing file: 2024 Waste Data Interrogator - Wastes Removed (Excel) - Version 2.xlsb
Available sheets: ['Information', 'Glossary', '2024 Waste Removed', 'Interrogator - Waste Removed', 'Active Sites', 'Configurable Reports', 'One Click Summaries', 'Waste Data Links', 'European Waste Catalogue', 'List of Changes']
Reading sheet: 2024 Waste Removed

Merge completed successfully.
Output file: /Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Data_clean/Merged_Waste_Data_Removed_